# Data Acquisition Pipeline — Nigeria

**Norman Lear Center × Gates Foundation — Manfluencer project**

Single notebook for all source-data acquisition for the Nigeria analysis. Two independent parts:

```
PART A — Transcribing Banky's MENtality podcast (6 episodes)
   audio (mp3) → YouTube captions → mlx-whisper ASR → pyannote diarisation
                 → Gemini speaker labelling → caption alignment → final transcripts

PART B — Scraping all tweets for the 4 Nigeria X creators
   Apify advanced-x-profile-scraper → per-creator xlsx with full engagement metrics
```

The two parts share the same `Setup` cell but are otherwise independent — run whichever you need.

---


# PART A — Transcription Pipeline (Banky MENtality)

## Setup

In [ ]:
from __future__ import annotations
import os, json, subprocess, re
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv

ROOT = Path.cwd().parents[1] if Path.cwd().name == "Notebooks" else Path.cwd()
load_dotenv(ROOT / ".env")
assert os.getenv("HF_TOKEN"), "HF_TOKEN required for pyannote diarization"
assert os.getenv("GEMINI_API_KEY"), "GEMINI_API_KEY required for refinement"
print("ROOT:", ROOT)


## Config — episodes + speaker panels

In [ ]:
AUDIO_DIR    = ROOT / "Nigeria" / "Content Analysis" / "Content - Raw" / "Banky Wellington" / "Audio Files"
CAPTIONS_DIR = ROOT / "Nigeria" / "Content Analysis" / "Content - Raw" / "Banky Wellington" / "Captions"
TRANS_DIR    = ROOT / "Nigeria" / "Content Analysis" / "Content - Raw" / "Banky Wellington" / "Transcripts"
TEMP_DIR     = ROOT / "temp" / "transcribe_mentality"
for d in (CAPTIONS_DIR, TRANS_DIR, TEMP_DIR):
    d.mkdir(parents=True, exist_ok=True)

# Each episode: audio file, YouTube ID, panel members (for Gemini speaker labeling)
EPISODES = [
    {
        "audio": "Masculinity + Money.mp3",
        "yt_id": "f6WW9g5hqLI",
        "panel": ["Ebuka Obi-Uchendu", "Banky W (Bankole Wellington)", "Seun Kuti", "Noble Igwe"],
    },
    {
        "audio": "Masculinity + Relationships.mp3",
        "yt_id": "mU5uAVhVEzA",
        "panel": ["Ebuka Obi-Uchendu", "Banky W", "Bovi Ugboma", "Do2dtun Energy gAD"],
    },
    {
        "audio": "Pt 2 Masculinity + Relationships.mp3",
        "yt_id": "7uLzlPGsiVo",
        "panel": ["Ebuka Obi-Uchendu", "Banky W", "Alex Ikemefuna", "Johnny Drille"],
    },
    {
        "audio": "Masculinity + Friendship.mp3",
        "yt_id": "XbFCPgdK8QQ",
        "panel": ["Ebuka Obi-Uchendu", "Banky W", "Alex Ikemefuna"],
    },
    {
        "audio": "Masculinity + Fatherhood.mp3",
        "yt_id": "V_eHJfW87iA",
        "panel": ["Ebuka Obi-Uchendu", "Banky W", "Timi Dakolo", "Hermes Iyele"],
    },
    {
        "audio": "Masculinity + Young Boys.mp3",
        "yt_id": "-YGXo00-fHw",
        "panel": ["Ebuka Obi-Uchendu", "Banky W", "IK Osakioduwa", "Murewa", "Sonariwo OnDeck"],
    },
]

WHISPER_MODEL = "mlx-community/whisper-large-v3-mlx"  # best quality on Apple Silicon
DIARIZATION_MODEL = "pyannote/speaker-diarization-3.1"
GEMINI_MODEL = "gemini-2.5-flash"

print(f"Episodes configured: {len(EPISODES)}")


## Step 1 — Download YouTube captions (ground truth)

Auto-generated and manually-uploaded captions from the original videos. These serve as the authoritative word source for the alignment step (eliminates whisper hallucination).


In [ ]:
def download_captions(yt_id, episode_audio_name):
    out_stem = Path(episode_audio_name).stem
    target = CAPTIONS_DIR / f"{out_stem}.txt"
    if target.exists():
        print(f"  cached: {target.name}")
        return target
    cmd = [
        "yt-dlp",
        "--write-auto-sub", "--sub-lang", "en", "--sub-format", "vtt",
        "--skip-download",
        "-o", str(TEMP_DIR / f"{out_stem}.%(ext)s"),
        f"https://www.youtube.com/watch?v={yt_id}",
    ]
    print(f"  downloading: {out_stem}")
    subprocess.run(cmd, capture_output=True, check=True)
    vtt = next(TEMP_DIR.glob(f"{out_stem}*.vtt"), None)
    if vtt is None:
        print(f"  ! no VTT for {yt_id}")
        return None
    # Strip VTT timestamps + tags into plain text
    lines = []
    seen = set()
    for line in vtt.read_text().splitlines():
        line = line.strip()
        if not line or "-->" in line or line.startswith(("WEBVTT", "Kind:", "Language:", "NOTE")):
            continue
        clean = re.sub(r"<[^>]+>", "", line).strip()
        if clean and clean not in seen:
            lines.append(clean)
            seen.add(clean)
    target.write_text("\n".join(lines))
    print(f"  saved {target.name} ({len(lines)} lines)")
    return target

for ep in EPISODES:
    download_captions(ep["yt_id"], ep["audio"])
print("\nCaptions ready.")


## Step 2 — Whisper ASR (mlx-whisper, word-level timestamps)

Uses `mlx-whisper` with the large-v3 model — best accuracy on Apple Silicon. Output is a JSON with word-level timestamps cached to `temp/`.


In [ ]:
import mlx_whisper

def transcribe(audio_path):
    cache = TEMP_DIR / f"{audio_path.stem}.whisper.json"
    if cache.exists():
        print(f"  cached: {cache.name}")
        return json.loads(cache.read_text())
    print(f"  transcribing {audio_path.name} ...")
    result = mlx_whisper.transcribe(
        str(audio_path),
        path_or_hf_repo=WHISPER_MODEL,
        word_timestamps=True,
        verbose=False,
    )
    cache.write_text(json.dumps(result, indent=1, default=str))
    print(f"  saved → {cache.name}  ({len(result.get('segments', []))} segments)")
    return result

for ep in EPISODES:
    transcribe(AUDIO_DIR / ep["audio"])


## Step 3 — Pyannote diarisation (speaker turns)

Runs `pyannote/speaker-diarization-3.1` per episode. Requires HF token + EULA accepted at https://hf.co/pyannote/speaker-diarization-3.1.


In [ ]:
from pyannote.audio import Pipeline
import torch

print("Loading pyannote pipeline (one-time, ~30s)...")
pipeline = Pipeline.from_pretrained(DIARIZATION_MODEL, token=os.getenv("HF_TOKEN"))
if torch.backends.mps.is_available():
    pipeline.to(torch.device("mps"))
    print("Using MPS (Apple Silicon)")

def diarize(audio_path):
    cache = TEMP_DIR / f"{audio_path.stem}.diarization.json"
    if cache.exists():
        print(f"  cached: {cache.name}")
        return json.loads(cache.read_text())
    print(f"  diarising {audio_path.name} ...")
    diarization = pipeline(str(audio_path))
    turns = [{"start": float(seg.start), "end": float(seg.end), "speaker": str(label)}
             for seg, _, label in diarization.itertracks(yield_label=True)]
    cache.write_text(json.dumps(turns, indent=1))
    print(f"  saved → {cache.name}  ({len(turns)} turns, {len({t['speaker'] for t in turns})} speakers)")
    return turns

for ep in EPISODES:
    diarize(AUDIO_DIR / ep["audio"])


## Step 4 — Merge whisper + diarisation into a labelled transcript

Aligns whisper word timestamps to pyannote turn boundaries, producing a transcript where each segment has a speaker label (`SPEAKER_00`, `SPEAKER_01`, etc.).


In [ ]:
def merge_whisper_diarization(whisper_result, turns):
    # Build a quick start-time → speaker lookup
    def speaker_at(t):
        for turn in turns:
            if turn["start"] <= t <= turn["end"]:
                return turn["speaker"]
        # fall back to nearest
        return min(turns, key=lambda x: min(abs(x["start"]-t), abs(x["end"]-t)))["speaker"]

    out_lines = []
    current_speaker = None
    current_text = []
    for seg in whisper_result.get("segments", []):
        spk = speaker_at(seg["start"])
        if spk != current_speaker:
            if current_speaker is not None:
                out_lines.append(f"{current_speaker}: {' '.join(current_text).strip()}")
            current_speaker = spk
            current_text = [seg["text"].strip()]
        else:
            current_text.append(seg["text"].strip())
    if current_speaker is not None:
        out_lines.append(f"{current_speaker}: {' '.join(current_text).strip()}")
    return "\n\n".join(out_lines)

for ep in EPISODES:
    audio = AUDIO_DIR / ep["audio"]
    whisper_data = json.loads((TEMP_DIR / f"{audio.stem}.whisper.json").read_text())
    turns = json.loads((TEMP_DIR / f"{audio.stem}.diarization.json").read_text())
    transcript = merge_whisper_diarization(whisper_data, turns)
    raw_path = TEMP_DIR / f"{audio.stem}.raw_labelled.txt"
    raw_path.write_text(transcript)
    print(f"  {audio.stem}: {len(transcript.split(chr(10)+chr(10)))} speaker turns → {raw_path.name}")


## Step 5 — Gemini speaker labelling (use panel context to assign real names)

Pass the raw `SPEAKER_00`/`01`/etc. transcript to Gemini along with the known panel members for each episode. Gemini will return the same transcript with proper names substituted (e.g., `SPEAKER_00 → Ebuka Obi-Uchendu`).


In [ ]:
from google import genai

client = genai.Client(api_key=os.getenv("GEMINI_API_KEY"))

LABEL_PROMPT = """You are a transcript editor. The transcript below uses generic labels (SPEAKER_00, SPEAKER_01, etc.) from automated diarisation. The actual speakers in this episode are listed below.

Your job: replace each generic label with the correct real name. Use voice/style/content cues — who tends to host, who introduces guests, who is being addressed by name, etc. Be consistent: the same SPEAKER_NN must always map to the same real name.

If you can't determine a label with high confidence, leave it as-is (do not guess wildly). Output ONLY the corrected transcript, nothing else.

Episode panel: {panel}

Transcript:
---
{transcript}
---"""

def gemini_label(raw_transcript, panel):
    prompt = LABEL_PROMPT.format(panel=", ".join(panel), transcript=raw_transcript[:120000])
    resp = client.models.generate_content(model=GEMINI_MODEL, contents=prompt)
    return resp.text.strip()

for ep in EPISODES:
    audio = AUDIO_DIR / ep["audio"]
    raw_path = TEMP_DIR / f"{audio.stem}.raw_labelled.txt"
    out_path = TRANS_DIR / f"{audio.stem}.txt"
    if out_path.exists():
        print(f"  cached: {out_path.name}")
        continue
    raw = raw_path.read_text()
    print(f"  Gemini-labelling {audio.stem} ...")
    labelled = gemini_label(raw, ep["panel"])
    out_path.write_text(labelled)
    print(f"  → {out_path.relative_to(ROOT)}")


## Step 6 — Verify each transcript

Quick sanity check: how many speakers Gemini ended up labelling, average segment length, total word count.


In [ ]:
import re
print(f"{'Episode':<45} {'Speakers':>10} {'Words':>10}")
print("-" * 70)
for ep in EPISODES:
    p = TRANS_DIR / f"{Path(ep['audio']).stem}.txt"
    text = p.read_text()
    speakers = set()
    words = 0
    for line in text.split("\n"):
        m = re.match(r"^([^:]+):\s+(.+)", line)
        if m:
            speakers.add(m.group(1).strip())
            words += len(m.group(2).split())
    print(f"{Path(ep['audio']).stem[:43]:<45} {len(speakers):>10} {words:>10}")


---
# PART B — Tweet Scraping (Nigeria X Creators)

Full historical scrape of every tweet for the 4 Nigeria X creators (Banky is YouTube-only, excluded).

| Creator | Handle | Orientation |
|---|---|---|
| Deyemi Okanlawon | `@_deyemi` | Progressive |
| Agba John Doe | `@jon_d_doe` | Regressive |
| Shola | `@itsSh0la` | Regressive |
| Wizarab | `@Wizarab10` | Regressive |

## Output

`Nigeria/Scraped Tweets/<Creator>_all_tweets.xlsx` — one file per creator, columns:
`tweet_link | text | likes | replies | retweets | quotes | bookmarks | views | timestamp`

Sorted newest-first, exact-text duplicates removed (keep highest-engagement copy).

## Backend

Apify actor `delicious_zebu/advanced-x-twitter-profile-scraper`. Requires `APIFY_API_KEY` in `.env`.

> **Note:** Free X scraping (snscrape, public Nitter, etc.) is essentially broken in 2025 due to Twitter API changes. Apify is the most reliable paid option.


## Setup

In [ ]:
import os
from pathlib import Path
import pandas as pd
from apify_client import ApifyClient
from dotenv import load_dotenv

ROOT = Path.cwd().parents[1] if Path.cwd().name == "Notebooks" else Path.cwd()
load_dotenv(ROOT / ".env")
APIFY_KEY = os.getenv("APIFY_API_KEY")
assert APIFY_KEY, "APIFY_API_KEY missing"

OUT_DIR = ROOT / "Nigeria" / "Scraped Tweets"
OUT_DIR.mkdir(parents=True, exist_ok=True)
CACHE_DIR = ROOT / "temp" / "scrape_all_tweets"
CACHE_DIR.mkdir(parents=True, exist_ok=True)
print("ROOT:", ROOT)


## Config

In [ ]:
CREATORS = [
    {"name": "Deyemi Okanlawon",  "handle": "_deyemi"},
    {"name": "Agba John Doe",     "handle": "jon_d_doe"},
    {"name": "Shola",             "handle": "itsSh0la"},
    {"name": "Wizarab",           "handle": "Wizarab10"},
]
MAX_PER_CREATOR = 5000
START_DATE = "2020-01-01"
END_DATE = "2026-12-31"


## Scrape — one Apify run per creator

Per-creator results are cached to parquet. Re-running uses the cache (no extra Apify cost).


In [ ]:
def scrape_creator(client, creator):
    cache_path = CACHE_DIR / f"raw_{creator['handle']}.parquet"
    if cache_path.exists():
        df = pd.read_parquet(cache_path)
        print(f"  [{creator['name']}] cache hit: {len(df)}")
        return df
    print(f"  [{creator['name']}] scraping x.com/{creator['handle']} ...")
    run = client.actor("delicious_zebu/advanced-x-twitter-profile-scraper").call(
        run_input={
            "accountUrls": [f"https://x.com/{creator['handle']}"],
            "maxCollections": MAX_PER_CREATOR,
            "language": "any",
            "splitMode": "month",
            "startDate": START_DATE,
            "endDate": END_DATE,
        },
        timeout_secs=3600,
    )
    items = list(client.dataset(run["defaultDatasetId"]).iterate_items())
    df = pd.DataFrame(items)
    if "authorHandle" in df.columns:
        df = df[df["authorHandle"].astype(str).str.lower() == creator["handle"].lower()].copy()
    df.to_parquet(cache_path, index=False)
    print(f"  [{creator['name']}] {len(df)} tweets saved")
    return df

client = ApifyClient(APIFY_KEY)
raw_per_creator = {c["name"]: scrape_creator(client, c) for c in CREATORS}


## Normalise + clean

Map the actor's column names to the canonical schema, dedup on text (keep highest-engagement copy), sort newest-first, write per-creator xlsx.


In [ ]:
COLUMN_MAP = [
    ("tweet_link",  ["tweetUrl", "url", "permanentUrl"]),
    ("text",        ["fullText", "text", "content"]),
    ("likes",       ["likeCount", "favoriteCount"]),
    ("replies",     ["replyCount"]),
    ("retweets",    ["repostCount", "retweetCount"]),
    ("quotes",      ["quoteCount"]),
    ("bookmarks",   ["bookmarkCount"]),
    ("views",       ["viewCount", "impressionCount"]),
    ("timestamp",   ["createdAt", "createdAtIso"]),
]

def normalise(df):
    out = pd.DataFrame()
    for out_col, candidates in COLUMN_MAP:
        for src in candidates:
            if src in df.columns:
                out[out_col] = df[src]
                break
        else:
            out[out_col] = None
    return out

summary = []
for creator in CREATORS:
    raw = raw_per_creator[creator["name"]]
    if raw.empty:
        continue
    out = normalise(raw)
    out["_dedupe_key"] = out["text"].astype(str).str.strip().str.lower()
    out = out.sort_values("likes", ascending=False).drop_duplicates("_dedupe_key", keep="first")
    out = out.drop(columns="_dedupe_key").reset_index(drop=True)
    if "timestamp" in out.columns:
        out["_ts"] = pd.to_datetime(out["timestamp"], errors="coerce", utc=True)
        out = out.sort_values("_ts", ascending=False).drop(columns="_ts").reset_index(drop=True)
    out_path = OUT_DIR / f"{creator['name']}_all_tweets.xlsx"
    out.to_excel(out_path, index=False)
    summary.append({"creator": creator["name"], "tweets": len(out)})
    print(f"  → {out_path.relative_to(ROOT)}: {len(out)} tweets")

pd.DataFrame(summary)
